# 🏦 Statement OCR to Google Sheets (PaddleOCR-VL-1.6 SOTA)
Extract bank statement transaction records into a styled **Google Sheet** with automated cell-level review highlighting.

### 🌟 Powered by PaddleOCR-VL-1.6:
- **Vision-Language Model (~0.9B VLM)** designed specifically for complex table parsing.
- Does **not** rely on rigid straight vertical lines (HoughLines), making it resilient against perspective skew, camera angle, folds, shadows, and blurred dividers.
- Outputs structured table data with automated validation checks.
- Also includes **PP-OCRv6** as an optional alternative.

### ⚡ Recommended Colab Setup:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run the steps sequentially below.

### Recovery after an initialization error
Restart the session to clear partial imports. If packages are already installed, skip Step 1 and run Steps 2–6. For a fresh runtime, run Step 1 once, restart, then run Steps 2–6.


In [ ]:
# @title Step 1: Install dependencies once, then restart the session
import os
import sys
import subprocess

os.environ["PADDLE_PDX_EAGER_INIT"] = "False"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

if any(name in sys.modules for name in ("paddle", "paddlex", "paddleocr")):
    raise RuntimeError("Restart the session before installing packages. If already installed, skip Step 1 and run Steps 2–6.")

subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "paddlepaddle", "paddlepaddle-gpu"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--timeout", "120",
                       "paddlepaddle-gpu==3.3.1", "-i",
                       "https://www.paddlepaddle.org.cn/packages/stable/cu126/"])
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "paddleocr[doc-parser]==3.7.0", "paddlex==3.7.0",
                       "openpyxl", "pillow", "gspread"])
print("Installation complete. Runtime > Restart session, then run Steps 2–6. Do not rerun Step 1.")


In [ ]:
# @title Step 2: Mount Google Drive & Authenticate Google Sheets
import os
from google.colab import drive

# 1. Mount Google Drive (only if not already mounted)
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=False)
else:
    print("✅ Google Drive is already mounted (skipping mount prompt).")

# 2. Authenticate Google Sheets (only if not already authenticated in this session)
if 'gc' not in globals() or gc is None:
    from google.colab import auth
    from google.auth import default
    import gspread
    print("Authenticating for Google Sheets (one-time per session)...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("✅ Authenticated & Google Sheets ready!")
else:
    print("✅ Google Sheets is already authenticated in this session! Skipping prompt.")


In [ ]:
# @title Step 3: Configuration
from pathlib import Path

# Path to the folder in Google Drive where you uploaded statement images
# Examples:
#   '/content/drive/MyDrive/statement_images'
#   'statement_images' (auto-resolves under /content/drive/MyDrive)
DRIVE_IMAGE_FOLDER = "/content/drive/MyDrive/statement_images" # @param {type:"string"}

# OCR Engine:
# - "PaddleOCR-VL-1.6": Vision-language model (~0.9B), with geometry-first crops and separate monetary-cell reading. (Default & Recommended on Colab GPU)
# - "PP-OCRv6": Lightweight geometric line-detection engine.
OCR_ENGINE = "PaddleOCR-VL-1.6" # @param ["PaddleOCR-VL-1.6", "PP-OCRv6"]

# Custom Title for the output Google Sheet (leave empty for auto timestamp)
CUSTOM_SHEET_TITLE = "test" # @param {type:"string"}

# Flag rows below this confidence score (0.0 to 1.0)
MIN_CONFIDENCE = 0.85 # @param {type:"number"}

# Also save a backup .xlsx file in the same Drive folder
SAVE_BACKUP_EXCEL = True # @param {type:"boolean"}

# Normalize folder path
image_dir = Path(DRIVE_IMAGE_FOLDER.strip())
if not image_dir.is_absolute():
    image_dir = Path("/content/drive/MyDrive") / image_dir

print(f"🚀 Selected Engine: {OCR_ENGINE}")
print(f"📁 Target image folder: {image_dir}")
if image_dir.exists():
    items = list(image_dir.iterdir())
    print(f"Folder exists! Found {len(items)} items.")
else:
    print(f"⚠️ Warning: Folder '{image_dir}' does not exist yet. Please check the path in Google Drive.")


In [ ]:
# @title Step 4: Core OCR Engines (PaddleOCR-VL-1.6 & PP-OCRv6)
import os
import sys
# Also update a flags module cached by an earlier failed import.
os.environ["PADDLE_PDX_EAGER_INIT"] = "False"
_cached_pdx_flags = sys.modules.get("paddlex.utils.flags")
if _cached_pdx_flags is not None:
    _cached_pdx_flags.EAGER_INITIALIZATION = False
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PADDLE_PDX_DISABLE_DEVICE_FALLBACK"] = "True"
import bisect
from html import unescape
import os
from pathlib import Path
import re
import statistics
import sys
from datetime import datetime
from uuid import uuid4
import cv2
import numpy as np
from PIL import Image, ImageOps

COLUMNS = ("index", "time", "date", "id", "content", "money_out", "money_in", "total_raw")
COLUMN_FRACTIONS = (0, 0.0362, 0.1114, 0.1877, 0.3842, 0.7239, 0.8216, 0.9066, 1)

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

ROW_ISSUE_COLUMNS = {
    "no_amount_detected": ("money_out", "money_in"),
    "both_amount_columns_populated": ("money_out", "money_in"),
    "index_gap_or_duplicate": ("index",),
}

def discover_images(folder):
    def natural_key(path):
        return tuple((1, int(part)) if part.isdigit() else (0, part.casefold())
                     for part in re.split(r"(\d+)", path.name))
    return sorted((p for p in folder.iterdir()
                   if p.is_file() and not p.name.startswith(".") and p.suffix.lower() in IMAGE_SUFFIXES),
                  key=natural_key)

def parse_html_table(html_str):
    """Extract rows and cells from an HTML table."""
    rows = []
    tr_matches = re.findall(r"<tr[^>]*>(.*?)</tr>", html_str, flags=re.DOTALL | re.IGNORECASE)
    for tr in tr_matches:
        cells = re.findall(r"<(?:td|th)[^>]*>(.*?)</(?:td|th)>", tr, flags=re.DOTALL | re.IGNORECASE)
        clean_cells = [unescape(re.sub(r"<[^>]+>", "", c)).strip() for c in cells]
        if any(clean_cells):
            rows.append(clean_cells)
    return rows

def validate_and_flag_records(records, min_confidence=0.80, default_score=0.95):
    """Validate records, flag issues, and mark specific columns for review."""
    issues = []
    review_columns = {}
    confidence = []

    for row_number, record in enumerate(records, 1):
        scores = {col: default_score if record.get(col) else None for col in COLUMNS}
        confidence.append(scores)

        # 1. Numeric index check
        if not record["index"].isdigit():
            issues.append({"row": row_number, "column": "index", "issue": "missing_or_non_numeric_index"})
            review_columns.setdefault(row_number, set()).add("index")

        # 2. Required text fields check
        for name in ("time", "date", "id", "content", "total_raw"):
            if not record[name]:
                issues.append({"row": row_number, "column": name, "issue": "no_text_detected"})
                review_columns.setdefault(row_number, set()).add(name)

        # 3. Amounts check
        if not record["money_out"] and not record["money_in"]:
            issues.append({"row": row_number, "issue": "no_amount_detected"})
            review_columns.setdefault(row_number, set()).update(ROW_ISSUE_COLUMNS["no_amount_detected"])
        if record["money_out"] and record["money_in"]:
            issues.append({"row": row_number, "issue": "both_amount_columns_populated"})
            review_columns.setdefault(row_number, set()).update(ROW_ISSUE_COLUMNS["both_amount_columns_populated"])

    # Unexpected letters or a time-like colon in financial values always need cell review.
    for row_number, record in enumerate(records, 1):
        for name in COLUMNS[-3:]:
            if record[name] and not re.fullmatch(r"[+-]?\d[\d,.\s]*", record[name]):
                issues.append({"row": row_number, "column": name, "issue": "non_numeric_amount"})
                review_columns.setdefault(row_number, set()).add(name)

    # 4. Continuity check
    indices = [int(r["index"]) if r["index"].isdigit() else None for r in records]
    for row, (previous, current) in enumerate(zip(indices, indices[1:]), 2):
        if previous is not None and current is not None and current != previous + 1:
            issues.append({"row": row, "issue": "index_gap_or_duplicate", "previous": previous, "current": current})
            review_columns.setdefault(row, set()).add("index")

    return {"rows": records, "confidence": confidence, "issues": issues, "review_columns": review_columns}

# --- Engine 1: PaddleOCR-VL-1.6 (Vision-Language) ---
def require_gpu():
    import paddle
    if not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:
        raise RuntimeError("No Paddle CUDA GPU available. Select T4 GPU and install the GPU package in Step 1.")
    paddle.set_device("gpu:0")
    probe = paddle.ones([2, 2])
    result = paddle.matmul(probe, probe)
    assert result.place.is_gpu_place(), "GPU check unexpectedly used CPU"
    assert float(result.sum().item()) == 8.0, "GPU computation check failed"
    print(f"GPU verified: {paddle.device.cuda.get_device_name(0)}; tensor computation passed.")
    return "gpu:0"

def create_vl_pipeline():
    device = require_gpu()
    from paddleocr import PaddleOCRVL
    print(f"Initializing PaddleOCR-VL-1.6 on {device}...")
    return PaddleOCRVL(pipeline_version="v1.6", device=device, use_layout_detection=False,
                       use_doc_orientation_classify=False, use_doc_unwarping=False,
                       use_chart_recognition=False, use_seal_recognition=False, use_queues=False)

def visual_row_edges(table, columns):
    """Locate transaction starts from printed index glyphs, before any OCR call."""
    left, right = columns[:2]
    margin = max(3, round((right - left) * 0.1))
    pixels = np.asarray(table.crop((left + margin, 0, right - margin, table.height)))
    gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
    ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, 31, 10)
    rules = cv2.morphologyEx(ink, cv2.MORPH_OPEN,
                             np.ones((max(25, right - left), 1), np.uint8))
    ink = cv2.bitwise_and(ink, cv2.bitwise_not(rules))
    _, _, stats, _ = cv2.connectedComponentsWithStats(ink)
    glyphs = [(y, h) for x, y, w, h, area in stats[1:]
              if area >= 6 and w >= 2 and h >= 4 and w >= h * 0.12
              and h <= max(12, (right - left) * 0.9)]
    if not glyphs:
        raise ValueError("No index glyphs found; cannot establish numeric cell rows safely")
    height = statistics.median(h for _, h in glyphs)
    groups = []
    for y, h in sorted(glyphs, key=lambda box: box[0] + box[1] / 2):
        center = y + h / 2
        if not groups or center - statistics.median(item[0] + item[1] / 2 for item in groups[-1]) > height * 0.65:
            groups.append([])
        groups[-1].append((y, h))
    starts = [max(0, round(min(y for y, _ in group) - height * 0.6)) for group in groups]
    edges = [0, *starts[1:], table.height]
    if any(bottom - top < height for top, bottom in zip(edges, edges[1:])):
        raise ValueError("Index row bands overlap; numeric row assignment needs review")
    return edges, height


def prepare_vl_table(image_path):
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
    native, layout = detect_table(image)
    native_columns = list(layout["columns"])
    width = max(native.width, 1600)
    height = round(native.height * width / native.width)
    columns = [round(fraction * width) for fraction in COLUMN_FRACTIONS]
    x = np.interp(np.arange(width), columns, native_columns).astype(np.float32)
    y = np.linspace(0, native.height - 1, height, dtype=np.float32)
    map_x, map_y = np.meshgrid(x, y)
    scaled = Image.fromarray(cv2.remap(np.asarray(native), map_x, map_y, cv2.INTER_CUBIC,
                                      borderMode=cv2.BORDER_REPLICATE))
    rows, glyph_height = visual_row_edges(scaled, columns)
    scale_y = (scaled.height - 1) / (native.height - 1)
    layout.update(columns=columns, rows=rows, native_columns=native_columns,
                  native_rows=[min(native.height, round(edge / scale_y)) for edge in rows],
                  glyph_height=glyph_height / scale_y)
    layout["native_rows"][-1] = native.height
    print(f"Geometry ready: {len(rows) - 1} row bands, 8 columns; table resized to {scaled.width} x {scaled.height}.")
    return native, scaled, layout


def numeric_cell_crops(native, layout):
    """Assign intact digit lines to physical cells; keep valid blank amounts blank."""
    rows = layout["native_rows"]
    crops, issues = {}, []
    glyph_height = layout["glyph_height"]
    for column in range(5, 8):
        left, right = layout["native_columns"][column:column + 2]
        strip = np.asarray(native.crop((left + 3, 0, right - 3, native.height)))
        gray = cv2.cvtColor(strip, cv2.COLOR_RGB2GRAY)
        ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 31, 10)
        rules = cv2.morphologyEx(ink, cv2.MORPH_OPEN,
                                 np.ones((max(25, round(glyph_height * 2.5)), 1), np.uint8))
        ink = cv2.bitwise_and(ink, cv2.bitwise_not(rules))
        _, _, stats, _ = cv2.connectedComponentsWithStats(ink)
        letters = [(y, h) for x, y, w, h, area in stats[1:]
                   if area >= 5 and w >= 1 and max(3, glyph_height * 0.25) <= h <= glyph_height * 2]
        lines = []
        for y, h in sorted(letters, key=lambda box: box[0] + box[1] / 2):
            center = y + h / 2
            if not lines or center - statistics.median(item[0] + item[1] / 2 for item in lines[-1]) > glyph_height * 0.65:
                lines.append([])
            lines[-1].append((y, h))
        assigned = {}
        for line in lines:
            top = min(y for y, _ in line)
            bottom = max(y + h for y, h in line)
            row = bisect.bisect_right(rows, (top + bottom) / 2) - 1
            if 0 <= row < len(rows) - 1:
                assigned.setdefault(row, []).append((top, bottom))
        for row, bands in assigned.items():
            top, bottom = min(b[0] for b in bands), max(b[1] for b in bands)
            if len(bands) > 1:
                issues.append({"row": row + 1, "column": COLUMNS[column], "issue": "multiple_numeric_lines_in_cell"})
            if top < rows[row] - 2 or bottom > rows[row + 1] + 2:
                issues.append({"row": row + 1, "column": COLUMNS[column], "issue": "numeric_box_crosses_row"})
            # Full cell width retains signs and punctuation; only trim empty vertical space.
            pixels = strip[max(0, top - 3):min(native.height, bottom + 3)]
            scale = min(3.0, max(1.0, 48 / pixels.shape[0]))
            if scale > 1:
                pixels = cv2.resize(pixels, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
            pixels = cv2.copyMakeBorder(pixels, 8, 8, 8, 8, cv2.BORDER_CONSTANT, value=(255, 255, 255))
            crops[(row, column)] = pixels[:, :, ::-1].copy()
    return crops, issues


def read_vl_numeric_cell(pixels, pipeline):
    texts = []
    # Reuse the existing VL model one cell at a time; no second OCR model or large crop batch.
    for result in pipeline.predict(input=pixels, use_layout_detection=False, prompt_label="ocr",
                                   use_doc_orientation_classify=False, use_doc_unwarping=False,
                                   use_queues=False, format_block_content=False,
                                   min_pixels=12544, max_pixels=200704, max_new_tokens=64):
        for block in result.get("parsing_res_list", []):
            content = getattr(block, "content", "") or (block.get("content", "") if isinstance(block, dict) else "")
            if content.strip():
                texts.append(unescape(re.sub(r"<[^>]+>", "", content)).strip())
    return " ".join(texts).strip()


def extract_with_vl_16(image_path, pipeline, min_confidence=0.80):
    native, table, layout = prepare_vl_table(image_path)
    numeric_crops, geometry_issues = numeric_cell_crops(native, layout)
    # Read only the five text columns as a table; financial fields have their own fixed cells.
    text_table = table.crop((0, 0, layout["columns"][5], table.height))
    output = pipeline.predict(input=np.asarray(text_table)[:, :, ::-1].copy(),
                              use_layout_detection=False, prompt_label="table",
                              use_doc_orientation_classify=False, use_doc_unwarping=False, use_queues=False)
    extracted_rows = []

    for res in output:
        # Check table_res_list
        table_list = res.get("table_res_list", [])
        if table_list:
            for table_res in table_list:
                html_code = ""
                if hasattr(table_res, "html") and isinstance(table_res.html, dict):
                    html_code = table_res.html.get("pred", "")
                elif isinstance(table_res, dict):
                    html_code = table_res.get("pred_html", "") or table_res.get("html", "")
                if html_code:
                    extracted_rows.extend(parse_html_table(html_code))

        # Check parsing_res_list blocks
        if not extracted_rows:
            blocks = res.get("parsing_res_list", [])
            for block in blocks:
                label = getattr(block, "label", None) or (block.get("label") if isinstance(block, dict) else "")
                content = getattr(block, "content", "") or (block.get("content") if isinstance(block, dict) else "")
                if label == "table" and content:
                    extracted_rows.extend(parse_html_table(content))

    if not extracted_rows:
        raise ValueError("PaddleOCR-VL-1.6 did not detect any table in this photo.")

    # Filter out header row if first row has non-digit index (e.g. 'STT', 'Ngày', etc.)
    if extracted_rows and extracted_rows[0][0].strip().casefold() in ("stt", "index", "no", "no.", "số tt", "số thứ tự"):
        extracted_rows = extracted_rows[1:]

    records = []
    for raw_row in extracted_rows:
        cells = (raw_row + [""] * 8)[:8]
        record = {name: cell.strip() for name, cell in zip(COLUMNS, cells)}
        for name in COLUMNS[-3:]:
            record[name] = ""  # These columns are read only from physical numeric cells.
        records.append(record)

    row_count = len(layout["rows"]) - 1
    if len(records) != row_count:
        # Preserve useful text, but never zip monetary cells to a different number of rows.
        result = validate_and_flag_records(records, min_confidence)
        for row in range(1, len(records) + 1):
            for column in COLUMNS[-3:]:
                result["issues"].append({"row": row, "column": column, "issue": "numeric_row_alignment_failed"})
                result["review_columns"].setdefault(row, set()).add(column)
        print(f"Row count mismatch: grid {row_count}, VL {len(records)}. Numeric columns flagged for review.")
        return result

    for row, record in enumerate(records):
        for column in range(5, 8):
            crop = numeric_crops.get((row, column))
            record[COLUMNS[column]] = read_vl_numeric_cell(crop, pipeline) if crop is not None else ""
        print(f"  Numeric cells: row {row + 1}/{row_count}", end="\r", flush=True)
    print()
    result = validate_and_flag_records(records, min_confidence)
    for row in range(1, len(records) + 1):
        for column in layout.get("inferred_columns", ()):
            geometry_issues.append({"row": row, "column": COLUMNS[column], "issue": "column_boundary_inferred"})
    result["issues"].extend(geometry_issues)
    for issue in geometry_issues:
        result["review_columns"].setdefault(issue["row"], set()).add(issue["column"])
    return result

# --- Engine 2: PP-OCRv6 Geometric ---
def create_v6_ocr():
    device = require_gpu()
    from paddleocr import PaddleOCR, TextRecognition
    print(f"Loading PP-OCRv6 on device: {device.upper()}...")
    ocr_kwargs = {
        "device": device,
        "text_detection_model_name": "PP-OCRv6_medium_det",
        "text_recognition_model_name": "PP-OCRv6_medium_rec",
        "use_doc_orientation_classify": False,
        "use_doc_unwarping": False,
        "use_textline_orientation": False,
        "text_rec_score_thresh": 0.0
    }
    rec_kwargs = {"model_name": "PP-OCRv6_medium_rec", "device": device}
    if device == "cpu":
        ocr_kwargs["enable_mkldnn"] = False
        ocr_kwargs["cpu_threads"] = 4
        rec_kwargs["enable_mkldnn"] = False
        rec_kwargs["cpu_threads"] = 4
    return PaddleOCR(**ocr_kwargs), TextRecognition(**rec_kwargs)

def rectify(image, corners):
    points = np.asarray(corners, dtype=np.float32)
    if points.shape != (4, 2) or not cv2.isContourConvex(points.astype(np.int32)):
        raise ValueError("Could not determine a convex table outline")
    width = round(max(np.linalg.norm(points[1] - points[0]), np.linalg.norm(points[2] - points[3])))
    height = round(max(np.linalg.norm(points[3] - points[0]), np.linalg.norm(points[2] - points[1])))
    if min(width, height) < 30:
        raise ValueError("Detected table is too small")
    target = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    matrix = cv2.getPerspectiveTransform(points, target)
    return Image.fromarray(cv2.warpPerspective(np.asarray(image), matrix, (width, height)))

def detect_table(image):
    scale = min(1, 1200 / max(image.size))
    photo = cv2.resize(np.asarray(image), (round(image.width * scale), round(image.height * scale)))
    height, width = photo.shape[:2]
    gray = cv2.cvtColor(photo, cv2.COLOR_RGB2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 31, 10)
    binary[cv2.cvtColor(photo, cv2.COLOR_RGB2HSV)[:, :, 1] > 80] = 0
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,
                             np.ones((max(5, height // 80), 1), np.uint8))
    lines = cv2.HoughLinesP(binary, 1, np.pi / 720, threshold=max(20, height // 27),
                           minLineLength=height // 10, maxLineGap=height // 30)
    candidates = []
    for x0, y0, x1, y1 in (() if lines is None else lines.reshape(-1, 4)):
        if abs(y1 - y0) > height * 0.15 and abs(x1 - x0) < abs(y1 - y0) * 0.25:
            middle = x0 + (height / 2 - y0) * (x1 - x0) / (y1 - y0)
            candidates.append((middle, (x0, y0, x1, y1)))
    groups = []
    for middle, line in sorted(candidates):
        if not groups or middle - statistics.mean(item[0] for item in groups[-1]) > max(3, width * 0.008):
            groups.append([])
        groups[-1].append((middle, line))
    if not 5 <= len(groups) <= 9:
        raise ValueError(f"Found {len(groups)} column boundaries; expected 9 for this eight-column statement.")
    rules, tops, bottoms = [], [], []
    for group in groups:
        points = np.array([(x, y) for _, (x0, y0, x1, y1) in group
                           for x, y in ((x0, y0), (x1, y1))], dtype=float)
        rules.append(np.polyfit(points[:, 1], points[:, 0], 1))
        tops.append(float(points[:, 1].min()))
        bottoms.append(float(points[:, 1].max()))
    middle_x = np.array([np.polyval(rule, height / 2) for rule in rules])

    def horizontal_edge(ends, padding):
        ends = np.array(ends)
        best, valid = (-1, float("-inf")), None
        for left in range(len(rules) - 1):
            for right in range(left + 1, len(rules)):
                edge = np.polyfit(middle_x[[left, right]], ends[[left, right]], 1)
                residual = np.abs(np.polyval(edge, middle_x) - ends)
                support = residual < height * 0.015
                score = (int(support.sum()), -float(residual[support].sum()))
                if score > best:
                    best, valid = score, support
        if best[0] < 5:
            raise ValueError("Table outline is unclear; use a photo showing the full table")
        edge = np.polyfit(middle_x[valid], ends[valid], 1)
        edge[1] += padding
        return edge

    top_edge = horizontal_edge(tops, -6)
    bottom_edge = horizontal_edge(bottoms, 6)

    def intersection(rule, edge):
        y = np.clip((edge[0] * rule[1] + edge[1]) / (1 - edge[0] * rule[0]), 0, height - 1)
        return float(np.polyval(rule, y) / scale), float(y / scale)

    corners = [intersection(rule, edge) for rule, edge in
               ((rules[0], top_edge), (rules[-1], top_edge),
                (rules[-1], bottom_edge), (rules[0], bottom_edge))]
    table = rectify(image, corners)
    target = np.float32([[0, 0], [table.width - 1, 0],
                         [table.width - 1, table.height - 1], [0, table.height - 1]])
    matrix = cv2.getPerspectiveTransform(np.float32(corners), target)
    points = np.float32([[[np.polyval(rule, height / 2) / scale,
                          height / 2 / scale]] for rule in rules[1:-1]])
    columns = [0, *[round(float(point[0])) for point in cv2.perspectiveTransform(points, matrix)[:, 0]], table.width]
    expected = np.asarray(COLUMN_FRACTIONS)
    observed = np.asarray(columns) / table.width
    matches = np.abs(observed[:, None] - expected).argmin(axis=1)
    if len(set(matches)) != len(matches) or np.max(np.abs(observed - expected[matches])) > 0.012:
        raise ValueError("Printed dividers do not match this statement's column template")
    missing = set(range(1, 8)) - set(matches)
    complete = [round(float(fraction) * table.width) for fraction in expected]
    for index, boundary in zip(matches, columns):
        complete[int(index)] = boundary
    if any(right - left < 7 for left, right in zip(complete, complete[1:])):
        raise ValueError("Detected columns are too narrow to crop safely")
    return table, {"image_size": list(image.size), "corners": corners, "columns": complete,
                   "inferred_columns": sorted({edge - 1 for edge in missing} | {edge for edge in missing})}

def detect_rows(column_results, height):
    anchors = []
    for column in range(3):
        for token in column_results[column]:
            text = token["text"].strip()
            if (column == 0 and text.isdigit()) or (column in (1, 2) and re.match(r"\d{1,2}/\d{1,2}/\d{4}", text)):
                anchors.append(token)
    if not anchors:
        raise ValueError("No transaction numbers or dates detected; cannot identify statement rows")
    line_height = min(statistics.median(t["box"][3] - t["box"][1] for t in anchors), height / 40)
    groups = []
    for token in sorted(anchors, key=lambda t: t["box"][1]):
        y = token["box"][1]
        if not groups or y - statistics.median(groups[-1]) > max(3, line_height * 0.75):
            groups.append([])
        groups[-1].append(y)
    starts = [max(0, round(statistics.median(group) - line_height * 0.35)) for group in groups]
    rows = [*starts, height]
    return rows

def read_tokens(image, ocr, scale=1):
    crop = np.asarray(image)[:, :, ::-1].copy()
    if scale > 1:
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    crop = cv2.copyMakeBorder(crop, 12, 12, 12, 12, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    tokens = []
    for prediction in ocr.predict(crop):
        for text, score, box in zip(prediction["rec_texts"], prediction["rec_scores"], prediction["rec_boxes"], strict=True):
            if text.strip():
                x0, y0, x1, y1 = (int(v) for v in box)
                tokens.append({"text": text, "score": float(score),
                               "box": [round((x0 - 12) / scale + 2), round((y0 - 12) / scale),
                                       round((x1 - 12) / scale + 2), round((y1 - 12) / scale)]})
    return tokens

def read_index_tokens(table, layout, recognizer):
    left, right = layout["columns"][:2]
    crops, boxes = [], []
    for top, bottom in zip(layout["rows"], layout["rows"][1:]):
        crop = np.asarray(table.crop((left + 3, top, right - 3, bottom)))[:, :, ::-1].copy()
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 31, 10)
        _, _, stats, _ = cv2.connectedComponentsWithStats(ink)
        letters = [(x, y, x + w, y + h) for x, y, w, h, area in stats[1:]
                   if area >= 5 and w >= 2 and h >= 4 and h < crop.shape[0] * 0.8 and h <= crop.shape[1]]
        if not letters:
            continue
        x0 = max(0, min(b[0] for b in letters) - 1)
        y0 = max(0, min(b[1] for b in letters) - 1)
        x1 = min(crop.shape[1], max(b[2] for b in letters) + 1)
        y1 = min(crop.shape[0], max(b[3] for b in letters) + 1)
        crops.append(crop[y0:y1, x0:x1])
        boxes.append([x0 + 3, y0 + top, x1 + 3, y1 + top])
    tokens = []
    if crops:
        for box, prediction in zip(boxes, recognizer.predict(crops, batch_size=8), strict=True):
            text = prediction["rec_text"].strip()
            if text:
                tokens.append({"text": text, "score": float(prediction["rec_score"]), "box": box})
    return tokens

def extract_with_v6(table, layout, threshold, ocr, number_recognizer):
    column_results = [[]]
    for column in range(1, len(COLUMNS)):
        left, right = layout["columns"][column:column + 2]
        image = table.crop((left + 2, 0, right - 2, table.height))
        column_results.append(read_tokens(image, ocr, scale=3 if image.width < 60 else 1))
    layout["rows"] = detect_rows(column_results, table.height)
    column_results[0] = read_index_tokens(table, layout, number_recognizer)

    cells = [[[] for _ in COLUMNS] for _ in range(len(layout["rows"]) - 1)]
    for column, tokens in enumerate(column_results):
        for token in tokens:
            x0, y0, x1, y1 = token["box"]
            row = bisect.bisect_right(layout["rows"], (y0 + y1) / 2) - 1
            if 0 <= row < len(cells):
                cells[row][column].append(token)

    records = []
    for row in cells:
        record = {}
        for name, tokens in zip(COLUMNS, row):
            tokens.sort(key=lambda t: (round(t["box"][1] / 5), t["box"][0]))
            record[name] = " ".join(t["text"] for t in tokens).strip()
        records.append(record)

    return validate_and_flag_records(records, threshold)


In [ ]:
# @title Step 5: Google Sheets & Excel Export Functions
def export_to_google_sheets(gc, results, failures, sheet_title):
    headers = list(COLUMNS) + ["flagged"]
    all_rows = [headers]
    review_cells = []

    for result in results:
        flags = {}
        for issue in result.get("issues", []):
            message = issue["issue"].replace("_", " ")
            if issue.get("column"):
                message = issue["column"] + ": " + message
            if issue["issue"] == "low_confidence":
                message += f" {issue['score']:.2f} < {issue['threshold']:.2f}"
            if "row" not in issue:
                message += ": " + issue.get("token", {}).get("text", "")
            flags.setdefault(issue.get("row"), []).append(message)

        review_columns = result.get("review_columns", {})

        for number, record in enumerate(result.get("rows", []), 1):
            current_row_idx = len(all_rows) # 0-based row index for Google Sheets API
            messages = flags.get(number, [])
            flagged = result["source"] + ": " + "; ".join(dict.fromkeys(messages)) if messages else ""
            row_data = [str(record.get(col) or "") for col in COLUMNS] + [flagged]
            all_rows.append(row_data)

            # Record affected cells for red/pink (#F4CCCC) highlight
            for affected_col in review_columns.get(number, ()):
                if affected_col in COLUMNS:
                    col_idx = COLUMNS.index(affected_col)
                    review_cells.append((current_row_idx, col_idx))

        if flags.get(None):
            all_rows.append([""] * len(COLUMNS) + [result["source"] + ": " + "; ".join(dict.fromkeys(flags[None]))])

    for failure in failures:
        all_rows.append([""] * len(COLUMNS) + [failure["source_file"] + ": " + failure["error"]])

    print(f"Creating Google Sheet: '{sheet_title}'...")
    sh = gc.create(sheet_title)
    ws = sh.sheet1
    try:
        ws.update_title("Statement Results")
    except Exception:
        pass

    num_rows = max(len(all_rows) + 50, 100)
    num_cols = max(len(headers), 10)
    try:
        ws.resize(rows=num_rows, cols=num_cols)
    except Exception:
        pass

    try:
        ws.update(range_name='A1', values=all_rows)
    except TypeError:
        ws.update('A1', all_rows)

    col_widths = [80, 180, 130, 180, 500, 150, 150, 150, 600]
    requests = []

    # 1. Custom column widths
    for i, width in enumerate(col_widths):
        requests.append({
            "updateDimensionProperties": {
                "range": {
                    "sheetId": ws.id,
                    "dimension": "COLUMNS",
                    "startIndex": i,
                    "endIndex": i + 1,
                },
                "properties": {"pixelSize": width},
                "fields": "pixelSize",
            }
        })

    # 2. Header styling: dark slate blue (#243746), bold white text, centered
    requests.append({
        "repeatCell": {
            "range": {
                "sheetId": ws.id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                "startColumnIndex": 0,
                "endColumnIndex": len(headers),
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": {"red": 36 / 255, "green": 55 / 255, "blue": 70 / 255},
                    "textFormat": {
                        "foregroundColor": {"red": 1.0, "green": 1.0, "blue": 1.0},
                        "bold": True,
                    },
                    "horizontalAlignment": "CENTER",
                    "verticalAlignment": "MIDDLE",
                }
            },
            "fields": "userEnteredFormat(backgroundColor,textFormat,horizontalAlignment,verticalAlignment)",
        }
    })

    # 3. Freeze row 1
    requests.append({
        "updateSheetProperties": {
            "properties": {
                "sheetId": ws.id,
                "gridProperties": {
                    "frozenRowCount": 1,
                },
            },
            "fields": "gridProperties.frozenRowCount",
        }
    })

    # 4. Highlight flagged rows with soft yellow (#FFF2CC)
    requests.append({
        "addConditionalFormatRule": {
            "rule": {
                "ranges": [{
                    "sheetId": ws.id,
                    "startRowIndex": 1,
                    "endRowIndex": len(all_rows),
                    "startColumnIndex": 0,
                    "endColumnIndex": len(headers),
                }],
                "booleanRule": {
                    "condition": {
                        "type": "CUSTOM_FORMULA",
                        "values": [{"userEnteredValue": '=$I2<>""'}],
                    },
                    "format": {
                        "backgroundColor": {"red": 255 / 255, "green": 242 / 255, "blue": 204 / 255},
                    },
                },
            },
            "index": 0,
        }
    })

    # 5. Cell-level highlight: soft pink/red (#F4CCCC) for specific problem cells
    for r_idx, c_idx in review_cells:
        requests.append({
            "repeatCell": {
                "range": {
                    "sheetId": ws.id,
                    "startRowIndex": r_idx,
                    "endRowIndex": r_idx + 1,
                    "startColumnIndex": c_idx,
                    "endColumnIndex": c_idx + 1,
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": {"red": 244 / 255, "green": 204 / 255, "blue": 204 / 255},
                    }
                },
                "fields": "userEnteredFormat.backgroundColor",
            }
        })

    try:
        sh.batch_update({"requests": requests})
    except Exception as e:
        print(f"Formatting note: {e}")

    return sh, len(all_rows) - 1

def export_to_excel(path, results, failures=()):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill

    workbook = Workbook()
    data = workbook.active
    data.title = "Data"
    data.append((*COLUMNS, "flagged"))
    review_fill = PatternFill("solid", fgColor="F4CCCC")

    for result in results:
        flags = {}
        for issue in result.get("issues", []):
            message = issue["issue"].replace("_", " ")
            if issue.get("column"):
                message = issue["column"] + ": " + message
            if issue["issue"] == "low_confidence":
                message += f" {issue['score']:.2f} < {issue['threshold']:.2f}"
            if "row" not in issue:
                message += ": " + issue.get("token", {}).get("text", "")
            flags.setdefault(issue.get("row"), []).append(message)

        review_columns = result.get("review_columns", {})

        for number, record in enumerate(result.get("rows", []), 1):
            messages = flags.get(number, [])
            flagged = result["source"] + ": " + "; ".join(dict.fromkeys(messages)) if messages else None
            data.append((*[record.get(name) or None for name in COLUMNS], flagged))

            for name in review_columns.get(number, ()):
                if name in COLUMNS:
                    data.cell(data.max_row, COLUMNS.index(name) + 1).fill = review_fill

            data.row_dimensions[data.max_row].height = min(160, max(30, 15 * (len(record.get("content", "")) // 75 + 1)))

        if flags.get(None):
            data.append((*[None for _ in COLUMNS], result["source"] + ": " + "; ".join(dict.fromkeys(flags[None]))))

    for failure in failures:
        data.append((*[None for _ in COLUMNS], failure["source_file"] + ": " + failure["error"]))

    data.freeze_panes = "A2"
    data.auto_filter.ref = data.dimensions
    data.row_dimensions[1].height = 24
    for cell in data[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="243746")
    for column, width in enumerate((9, 24, 16, 24, 78, 20, 20, 20, 100), 1):
        data.column_dimensions[data.cell(1, column).column_letter].width = width
    for row in data.iter_rows(min_row=2):
        for cell in row:
            cell.data_type = "s"
            cell.number_format = "@"
            cell.alignment = Alignment(vertical="top", wrap_text=True)
        if row[-1].value:
            row[-1].fill = PatternFill("solid", fgColor="FFF2CC")
    workbook.save(path)


In [ ]:
# @title Step 6: Run OCR Pipeline & Generate Google Sheet
from IPython.display import display, Markdown

if not image_dir.exists():
    raise FileNotFoundError(f"Folder not found: {image_dir}. Please upload images to Google Drive and update DRIVE_IMAGE_FOLDER in Step 3.")

images = discover_images(image_dir)
if not images:
    raise FileNotFoundError(f"No supported images (.jpg, .png, etc.) found in {image_dir}")

print(f"Found {len(images)} images to process in: {image_dir}\n")

# Initialize or reuse selected engine
if OCR_ENGINE == "PaddleOCR-VL-1.6":
    if 'vl_pipeline' not in globals() or vl_pipeline is None:
        vl_pipeline = create_vl_pipeline()
    else:
        print("⚡ Using already loaded PaddleOCR-VL-1.6 pipeline (skipping reload)...")
else:
    if 'ocr_v6' not in globals() or ocr_v6 is None:
        ocr_v6, number_recognizer = create_v6_ocr()
    else:
        print("⚡ Using already loaded PP-OCRv6 engine (skipping reload)...")

results, failures = [], []

for number, path in enumerate(images, 1):
    print(f"[{number}/{len(images)}] Processing: {path.name}...")
    try:
        if OCR_ENGINE == "PaddleOCR-VL-1.6":
            result = extract_with_vl_16(path, vl_pipeline, MIN_CONFIDENCE)
        else:
            with Image.open(path) as src:
                img = ImageOps.exif_transpose(src).convert("RGB")
            table, current = detect_table(img)
            result = extract_with_v6(table, current, MIN_CONFIDENCE, ocr_v6, number_recognizer)

        result["source"] = path.name
        results.append(result)
        print(f"  -> Extracted {len(result['rows'])} rows.")
    except Exception as error:
        print(f"  ❌ Failed {path.name}: {error}")
        failures.append({"source_file": path.name, "error": f"Failed: {error}"})

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
sheet_title = CUSTOM_SHEET_TITLE.strip() or f"Statement_OCR_Results_{timestamp}"

# 1. Export to Google Sheets
sh, total_rows = export_to_google_sheets(gc, results, failures, sheet_title)

# 2. Optional: Export backup Excel in the Drive folder
if SAVE_BACKUP_EXCEL:
    excel_path = image_dir / f"{sheet_title}.xlsx"
    export_to_excel(excel_path, results, failures)
    print(f"📁 Backup Excel saved to: {excel_path}")

flagged_rows = sum(len({issue["row"] for issue in result.get("issues", []) if "row" in issue}) for result in results)

# Display summary & clickable Google Sheet link
display(Markdown(f"""
---
### 🚀 Process Complete!
- **OCR Engine Used:** `{OCR_ENGINE}`
- **Total Images:** {len(images)} (Processed: {len(results)}, Failed: {len(failures)})
- **Total Data Rows:** {total_rows}
- **Flagged Rows for Review:** {flagged_rows}

### 🔗 Open Your Google Sheet:
👉 **[{sh.title}]({sh.url})**
---
"""))
